# Step 2: Integrate nomic-embed-text Embeddings (Ollama)

In [ ]:
import requests
import json
import numpy as np

# Ollama API endpoint
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_EMBED_MODEL = "nomic-embed-text"

def get_embedding(text: str) -> list:
    """Get embedding for text using Ollama's nomic-embed-text model."""
    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/embed",
            json={
                "model": OLLAMA_EMBED_MODEL,
                "input": text
            }
        )
        response.raise_for_status()
        embeddings = response.json()["embeddings"]
        return embeddings[0] if embeddings else None
    except Exception as e:
        print(f"❌ Error getting embedding: {e}")
        print(f"Make sure Ollama is running: ollama serve")
        return None

# Test: Check if Ollama is running
print("🔍 Testing Ollama connection...")
try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    models = response.json().get("models", [])
    print(f"✅ Ollama is running with {len(models)} model(s)")
    print(f"Available models: {[m['name'] for m in models]}")
except Exception as e:
    print(f"❌ Ollama not running: {e}")

## Test 1: Embed Sample Support Tickets

In [ ]:
# Sample support tickets (fake inbox data)
support_tickets = [
    "Customer unable to login with correct password",
    "How do I reset my password?",
    "Payment failed with invalid card error",
    "How can I update my billing information?",
    "App crashes when uploading large files",
    "Is there a mobile app available?",
    "How do I export my data?",
    "Bug: Notifications not working properly",
    "Can I upgrade my subscription plan?",
    "What payment methods do you accept?"
]

print(f"📝 Embedding {len(support_tickets)} sample support tickets...\n")

# Embed each ticket and store results
ticket_embeddings = {}
for i, ticket in enumerate(support_tickets, 1):
    embedding = get_embedding(ticket)
    if embedding:
        ticket_embeddings[ticket] = embedding
        print(f"✅ [{i}/{len(support_tickets)}] Embedded: '{ticket[:50]}...'")
        print(f"   Embedding dimension: {len(embedding)}")
    else:
        print(f"❌ [{i}/{len(support_tickets)}] Failed: '{ticket}'")

print(f"\n✅ Successfully embedded {len(ticket_embeddings)}/{len(support_tickets)} tickets")

## Test 2: Semantic Similarity Search

Now let's test if the embeddings can find similar tickets (without using any LLM yet)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_tickets(query: str, top_k: int = 3) -> list:
    """Find most similar tickets to a query."""
    if not ticket_embeddings:
        print("❌ No embeddings available. Run previous cell first.")
        return []
    
    # Embed the query
    query_embedding = get_embedding(query)
    if query_embedding is None:
        return []
    
    # Calculate similarity with all tickets
    query_embedding = np.array(query_embedding).reshape(1, -1)
    similarities = []
    
    for ticket, embedding in ticket_embeddings.items():
        embedding = np.array(embedding).reshape(1, -1)
        similarity = cosine_similarity(query_embedding, embedding)[0][0]
        similarities.append((ticket, similarity))
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Test query
query = "I forgot my password"
print(f"🔍 Query: '{query}'\n")
print("Top 3 similar support tickets:")
print("-" * 60)

similar = find_similar_tickets(query, top_k=3)
for i, (ticket, score) in enumerate(similar, 1):
    print(f"{i}. [{score:.3f}] {ticket}")

print("\n" + "=" * 60)

# Test another query
query2 = "How do I add a new payment method?"
print(f"\n🔍 Query: '{query2}'\n")
print("Top 3 similar support tickets:")
print("-" * 60)

similar2 = find_similar_tickets(query2, top_k=3)
for i, (ticket, score) in enumerate(similar2, 1):
    print(f"{i}. [{score:.3f}] {ticket}")

## Test 3: Embedding Statistics

In [ ]:
import pandas as pd

# Analyze embeddings
if ticket_embeddings:
    # Get first embedding for dimension info
    first_embedding = list(ticket_embeddings.values())[0]
    
    stats = {
        "Total Tickets Embedded": len(ticket_embeddings),
        "Embedding Dimension": len(first_embedding),
        "Model": OLLAMA_EMBED_MODEL,
        "Running Locally": "✅ Yes (Ollama)",
        "API Key Required": "❌ No",
        "Cost": "💰 Free ($0)",
    }
    
    print("📊 Embedding Statistics:")
    print("=" * 50)
    for key, value in stats.items():
        print(f"{key:.<30} {value}")
    
    # Show embedding sample
    print("\n📈 Sample Embedding (first 10 dimensions):")
    print("-" * 50)
    sample_embedding = first_embedding[:10]
    print(f"[{', '.join([f'{x:.4f}' for x in sample_embedding])}...]")
    
    # Embedding distribution stats
    embeddings_array = np.array(list(ticket_embeddings.values()))
    print("\n📉 Embedding Distribution:")
    print("-" * 50)
    print(f"Mean value:     {embeddings_array.mean():.6f}")
    print(f"Std deviation:  {embeddings_array.std():.6f}")
    print(f"Min value:      {embeddings_array.min():.6f}")
    print(f"Max value:      {embeddings_array.max():.6f}")
else:
    print("❌ No embeddings to analyze. Run previous cells first.")